# fitcfg — Colab 측정·검증 노트북 (Phase 2 · 3 · 7)

1. **측정**: 소형 모델(Llama-3.2-1B, Qwen2.5-3B)로 설정을 바꿔가며 peak GPU 메모리·처리량을 자동 측정
2. **보정**: 측정 CSV로 cost model 보정 + 예측 정확도(MAPE) 리포트
3. **실행 검증**: fitcfg가 추천한 학습 번들과 HF Trainer 기본값 번들을 실제로 돌려 OOM 여부·예측 오차 확인

위에서부터 순서대로 실행하면 됩니다. 2·3장만 설정하면 나머지는 그대로 실행하면 됩니다.

**시작 전 체크리스트**
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** (L4/A100이면 더 좋음)
2. 코드 준비 — 둘 중 하나
   - **Drive(권장, 결과가 안 날아감)**: `fitcfg` 폴더를 `내 드라이브/capstone/fitcfg`에 업로드
   - **업로드**: `fitcfg.zip`(프로젝트 폴더에 있음, 또는 `python scripts/pack.py`)을 1장 셀에서 업로드
3. Llama-3.2-1B는 HF **gated 모델**:
   - 방법 A: HF에서 라이선스 동의 → 토큰 발급 → Colab 왼쪽 🔑(Secrets)에 `HF_TOKEN` 등록, 노트북 액세스 허용
   - 방법 B: 아래 설정에서 `USE_MIRROR = True` → 동일 가중치 미러(`unsloth/Llama-3.2-1B`) 사용

**소요 시간(T4 기준 대략)**: smoke 1~2분 · `colab-core` 약 40~60분 · `colab-qwen3b`+`colab-lora-shape` 약 30~40분 · 실행 검증 약 15~25분.
끊겨도 괜찮습니다. **다시 실행하면 이미 측정한 설정은 건너뛰고 이어서** 진행합니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. 코드 준비

In [ ]:
CODE_SOURCE = "drive"      # "drive" (권장) 또는 "upload" (fitcfg.zip 업로드, 결과는 마지막에 다운로드)
DRIVE_REPO = "/content/drive/MyDrive/capstone/fitcfg"   # ← Drive에 올린 위치

import os, zipfile
if CODE_SOURCE == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = DRIVE_REPO
else:
    from google.colab import files
    up = files.upload()                                  # fitcfg.zip 선택
    zipfile.ZipFile(next(iter(up))).extractall("/content")
    REPO = "/content/fitcfg"
assert os.path.exists(f"{REPO}/scripts/measure.py"), f"{REPO} 에 fitcfg가 없습니다. 경로/파일을 확인하세요."
os.chdir(REPO)
print("작업 폴더:", os.getcwd())

In [ ]:
# 측정 의존성. torch는 Colab 기본 설치본을 그대로 사용 (재설치하면 오래 걸리고 CUDA 충돌 위험)
!pip install -q -U transformers peft bitsandbytes accelerate datasets
import torch, transformers, peft, bitsandbytes
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)
# ⚠️ 이 노트북 커널에서는 torch.cuda.* 를 호출하지 마세요.
#    커널이 GPU 컨텍스트(~0.3GB)를 잡으면 측정 프로세스의 메모리 수치가 오염됩니다.
#    (measure.py 가 시작 시점의 다른 프로세스 사용량을 빼서 보정하긴 하지만, 깨끗한 게 최선)

## 2. 설정

In [ ]:
USE_MIRROR = False            # HF gated 권한이 없으면 True
STEPS = 15                    # 설정당 학습 스텝 (앞 3스텝은 워밍업으로 처리량 계산에서 제외)
OUT = "data/measurements.csv" # Drive에 저장됨 → 세션이 끊겨도 유지

# HF 토큰 (Colab Secrets 의 HF_TOKEN)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN 설정됨")
except Exception as e:
    print("HF_TOKEN 없음 →", "미러 사용" if USE_MIRROR else "gated 모델 다운로드가 실패할 수 있습니다", f"({type(e).__name__})")

import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
TAG = "colab-" + gpu.replace("Tesla ", "").replace("NVIDIA ", "").split()[0].lower()
MIRROR = "--hf-override llama-3.2-1b=unsloth/Llama-3.2-1B" if USE_MIRROR else ""
EXTRA = f"--steps {STEPS} --out {OUT} --tag {TAG} {MIRROR}"
print("TAG:", TAG, "| 옵션:", EXTRA)

In [ ]:
!nvidia-smi --query-gpu=memory.used --format=csv   # 측정 전 GPU 사용량: 0에 가까워야 정상

## 3. 스모크 테스트 (1개 설정, 1~2분)
`OK device_peak=...` 가 나오면 성공. 실패하면 아래 **문제 해결**을 보세요.

In [ ]:
!python scripts/sweep.py --preset smoke --steps 5 --out data/smoke.csv --tag {TAG} {MIRROR}

## 4. 본 측정
먼저 `--dry-run`으로 목록과 **v0 예측 메모리**를 확인합니다. 예측이 GPU 용량의 1.5배를 넘는 설정은 자동으로 건너뛰고,
1.0~1.5배 사이는 일부러 실행해서 **OOM 데이터**도 모읍니다 (OOM 회피율 평가에 필요).

In [ ]:
!python scripts/sweep.py --preset colab-core colab-qwen3b colab-lora-shape --dry-run {EXTRA}

In [ ]:
# ★ 핵심: Llama-3.2-1B × (QLoRA / LoRA / Full) × batch·seq·checkpointing
!python scripts/sweep.py --preset colab-core {EXTRA}

In [ ]:
# 2번째 모델 + LoRA 모양 변화 (activation 항과 logits 항을 분리하는 데 필요)
!python scripts/sweep.py --preset colab-qwen3b colab-lora-shape {EXTRA}

## 5. 결과 확인

In [ ]:
import pandas as pd
df = pd.read_csv(OUT)
print(f"총 {len(df)}행 | 성공 {int((df.ok == 1).sum())} | OOM {int((df.note == 'OOM').sum())} | 기타 실패 {int(((df.ok == 0) & (df.note != 'OOM')).sum())}")
cols = ["model", "method", "batch", "seq", "ckpt", "optim", "lora_rank", "lora_targets",
        "device_peak_gb", "peak_alloc_gb", "mem_model_gb", "tokens_per_s", "note"]
df[cols].sort_values(["model", "method", "device_peak_gb"])

In [ ]:
# 실패(OOM 제외) 행이 있으면 원인 확인
df[(df.ok == 0) & (df.note != "OOM")][["model", "method", "batch", "seq", "ckpt", "note"]]

## 6. 보정 (Phase 3) — GPU 없이도 되지만 여기서 바로 확인 가능
보정 전(문헌 상수) vs 보정 후, held-out(LOO / 모델 제외) MAPE, OOM 판정 정확도를 출력합니다.
결과: `data/calibration.json`, `reports/calibration_report.md`, `reports/calibration_plot.png`

In [ ]:
import os
RUNS = "--runs data/bundle_runs.jsonl" if os.path.exists("data/bundle_runs.jsonl") else ""
!python scripts/calibrate.py --csv {OUT} {RUNS}

In [ ]:
from IPython.display import Markdown, Image, display
display(Image("reports/calibration_plot.png"))
display(Markdown(open("reports/calibration_report.md", encoding="utf-8").read()))

## 7. 실행 검증 (Phase 7) — 추천 번들 vs HF Trainer 기본값
fitcfg가 만든 **학습 번들을 실제로** 30스텝 돌립니다. 같은 조건에서 HF Trainer 기본값(micro-batch 8, checkpointing off)
번들도 돌려서 **"기본값은 OOM, 추천은 성공"** 을 실측으로 확인하고, 추천 설정의 예측 메모리 오차를 잽니다.
데이터: 공개 데이터셋 `tatsu-lab/alpaca` 앞 2,000개 (text 필드). 결과는 `data/bundle_runs.jsonl`.

> 6장에서 `data/calibration.json`이 생겼다면 번들 생성 시 **보정된 예측**이 쓰입니다 (더 정확).

In [ ]:
import os, subprocess
if USE_MIRROR:
    os.environ["FITCFG_MODEL_OVERRIDE"] = "unsloth/Llama-3.2-1B"   # llama 번들만 사용
SCENARIOS = [   # (이름, 모델, method, seq_len) — T4 15GB 기준
    ("qlora_1b_s1024", "llama-3.2-1b", "qlora", 1024),
    ("lora_1b_s2048", "llama-3.2-1b", "lora", 2048),
]
DATA = "--dataset tatsu-lab/alpaca --text-field text --dataset-split 'train[:2000]'"
for name, model, method, seq in SCENARIOS:
    for label, flag in [("recommended", ""), ("naive", "--naive")]:
        out = f"runs/{name}_{label}"
        r = subprocess.run(f"python -m fitcfg plan --model {model} --gpu t4 --method {method} --seq {seq} "
                           f"--top 1 --out {out} {flag} {DATA}", shell=True, capture_output=True, text=True)
        print((r.stdout + r.stderr).strip().splitlines()[-1])

In [ ]:
# 각 번들을 30스텝씩 실행 (기본값 번들은 대부분 곧바로 OOM으로 끝남)
for name, *_ in SCENARIOS:
    for label in ["recommended", "naive"]:
        !python scripts/run_bundle.py runs/{name}_{label} --label {label} --max-steps 30

In [ ]:
!python scripts/evaluate.py --runs data/bundle_runs.jsonl > /dev/null
from IPython.display import Markdown, display
md = open("reports/evaluation.md", encoding="utf-8").read()
display(Markdown(md[md.find("## 실제 실행 검증"):] if "## 실제 실행 검증" in md else md))

In [ ]:
# (업로드 모드일 때) 결과 파일 내려받기 — Drive 모드면 이미 Drive에 저장돼 있음
if CODE_SOURCE == "upload":
    import shutil
    shutil.make_archive("/content/fitcfg_results", "zip", REPO, "data")
    from google.colab import files
    files.download("/content/fitcfg_results.zip")

## 다음 단계
- `data/measurements.csv`와 `data/bundle_runs.jsonl`(Drive에 있음)을 공유해 주세요 → 잔차를 보고 cost model 식 자체를 같이 고칩니다.
- Kaggle(P100/T4×2)이나 다른 GPU에서도 같은 노트북을 쓰면 `gpu_name`으로 구분돼 한 CSV에 쌓입니다.

## 문제 해결
| 증상 | 원인 / 해결 |
|---|---|
| `401` / `gated repo` | HF 토큰·라이선스 동의 확인, 또는 `USE_MIRROR = True` |
| 모든 행이 `CRASH rc=...` | 스모크 셀 출력을 보고 에러 확인 (패키지 버전 문제일 가능성) |
| `bitsandbytes` 관련 에러 | `!pip install -U bitsandbytes` 후 **런타임 다시 시작** |
| 세션이 끊김 | 1·2장 셀을 다시 실행한 뒤 4장 셀 재실행 → 이어서 진행 |
| 번들 실행이 `ERROR` | `runs/<이름>/` 에서 `bash launch.sh` 를 직접 실행해 에러 확인 |
| 한 설정이 너무 오래 걸림 | `sweep.py --timeout` (기본 1200초) 초과 시 `TIMEOUT`으로 기록하고 다음으로 넘어감 |